# Fine-tune SQLCoder-7B-2 using QLoRA on Google Colab

This notebook guides you through dataset conversion and fine-tuning `defog/sqlcoder-7b-2` on the Spider Text-to-SQL dataset. It utilizes 4-bit NF4 quantization and PEFT to run on a single A100 GPU.

### Recommended Runtime
- Google Colab A100 GPU (24GB or 40GB VRAM)
- Python 3.10+

## 1. Install Dependencies

In [ ]:
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install -q -U git+https://github.com/huggingface/accelerate.git
!pip install -q -U datasets bitsandbytes trl pyyaml huggingface_hub

## 2. Authenticate with HuggingFace Hub

You will need a Hugging Face user access token (write access) to pull the base model and push the adapter weights.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Clone Repository or Copy Scripts

Clone the repository to get the training scripts:

In [ ]:
# Replace with your actual repository URL
!git clone https://github.com/your-username/text-to-sql-ai.git
%cd text-to-sql-ai/training

## 4. Download Spider Dataset

Download the dataset zip and extract it:

In [ ]:
!wget https://raw.githubusercontent.com/taoyihua/spider/master/spider.zip -O spider.zip
!unzip -q spider.zip -d ./spider_dataset

## 5. Convert Spider to SQLCoder Prompt Format

In [ ]:
!python scripts/convert_spider_to_sqlcoder.py \
    --spider_json ./spider_dataset/spider/train.json \
    --tables_json ./spider_dataset/spider/tables.json \
    --output_jsonl ./dataset_train_sqlcoder.jsonl

## 6. Run QLoRA Fine-Tuning

Run the training script. This script loads `defog/sqlcoder-7b-2`, wraps it with LoRA parameters, and trains it on the converted Spider dataset. It outputs the checkpoints to `./adapter_weights`.

In [ ]:
!python scripts/train_qlora.py \
    --config configs/qlora_sqlcoder_spider.yaml \
    --dataset ./dataset_train_sqlcoder.jsonl \
    --output_adapter ./adapter_weights

## 7. Push Trained Adapter to HuggingFace Hub

In [ ]:
# Set your repository name
HF_USERNAME = "your-username"
REPO_NAME = "sqlcoder-7b-2-spider-adapter"
REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"

!python scripts/push_adapter.py \
    --adapter_dir ./adapter_weights \
    --repo_id $REPO_ID